# Восстановление пунктуации (мультимодальное, late fusion) — корпус Russian LibriSpeech (RuLS)

Модель преобразует **текст без пунктуации → текст с пунктуацией**, опираясь *одновременно* на текст и на акустические признаки из звука (паузы, длительности, темп, **F0**, энергия). Восстанавливаются **запятые, точки, многоточия, вопросительные и восклицательные знаки**, а также **абзацы (красные строки)** и **капитализация**.


## Архитектуры
BiLSTM (baseline) · Transformer с нуля (baseline-трансформер) · RuBERT-base и rubert-tiny2 (предобученные). Все двухпоточные: текст ⊕ акустика → 3 головы.

## 0. Зависимости
Раскомментируйте при первом запуске.

In [1]:
# --- УСТАНОВКА (раскомментируйте при первом запуске) ---
# import sys
# Зависимости проекта:
# !{sys.executable} -m pip install -r requirements.txt
#
# Forced aligner (акустика: паузы/F0). ВАЖНО: ставьте ТЕМ ЖЕ python, что у ядра,
# иначе ядро его не увидит. ffmpeg обязателен.
# !{sys.executable} -m pip install git+https://github.com/MahmoudAshraf97/ctc-forced-aligner.git
# Windows: ffmpeg через conda -> conda install -c conda-forge ffmpeg
# Linux:   sudo apt install ffmpeg

import os
os.environ.setdefault("DATASETS_AUDIO_BACKEND", "soundfile")

'soundfile'

## 1. Импорт модулей

In [2]:
import numpy as np
import torch
from functools import partial
from torch.utils.data import DataLoader

from modules import (
    get_config,
    build_examples, WordVocab,
    BaselineDataset, PretrainedDataset, baseline_collate, pretrained_collate,
    build_model, load_hf_tokenizer,
    train_model, set_seed,
    evaluate, pretty_report,
    PunctuationRestorer, STTPunctuationPipeline,
    PRETRAINED_PRESETS, PUNCT_LABELS, PARA_LABELS, CAP_LABELS, ACOUSTIC_FEATURES,
)

cfg = get_config()
set_seed(cfg.train.seed)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
cfg.train.device = DEVICE
print("device:", DEVICE)
print("пунктуация:", PUNCT_LABELS, "| абзац:", PARA_LABELS, "| капитализация:", CAP_LABELS)
print("акустика:", ACOUSTIC_FEATURES)
print("loss:", cfg.train.loss_type, "| автовеса классов:", cfg.train.auto_class_weights)

device: cpu
пунктуация: ['O', 'COMMA', 'PERIOD', 'QUESTION', 'EXCLAM', 'ELLIPSIS'] | абзац: ['NO_PARA', 'PARA'] | капитализация: ['LOWER', 'CAP', 'UPPER']
акустика: ['pause_before', 'pause_after', 'word_duration', 'speech_rate', 'f0_end_median', 'f0_end_slope', 'energy_end']
loss: focal | автовеса классов: True


## 1b. Диагностика окружения

In [3]:
import modules
modules.diagnose()

modules.__version__ = 3.2-ruls
python (ядро)       = c:\Users\Roman\Documents\Projects\STT_russian_lang\venv\Scripts\python.exe
ruls_repo           = istupakov/russian_librispeech
text_field[0]       = text_no_preprocessing
  ок: версия модулей актуальная (RuLS, поле text_no_preprocessing).

forced-aligner:
  ок: ctc_forced_aligner импортируется (c:\Users\Roman\Documents\Projects\STT_russian_lang\venv\Lib\site-packages\ctc_forced_aligner\__init__.py)

датасеты:
  ок: datasets 2.20.0

RuLS:
  Основной путь — официальный архив OpenSLR (зеркала US/EU/CN), не HF Hub.
  Рекомендуется: python prepare_ruls.py  (см. README), затем examples_from_pkl(...).
  huggingface_hub доступен (можно пробовать и HF-путь).


## 2. Данные: Russian LibriSpeech (RuLS)


### Рекомендованный способ загрузки (если HuggingFace не сработал)

Загрузка https://www.openslr.org/96/
```bash
# в терминале, из папки проекта, тем же python что у ядра:
python prepare_ruls.py --limit 8000
# или, если архив скачали вручную браузером:
python prepare_ruls.py --no-download --limit 8000
```

In [4]:
# #  ЗАГРУЗКА ИЗ PKL (рекомендуется при недоступном HF) 
# from modules import examples_from_pkl
# train_examples = examples_from_pkl('ruls_examples.pkl', cfg.data,
#                                    use_alignment=USE_ALIGNMENT, split='train')
# val_examples   = examples_from_pkl('ruls_examples.pkl', cfg.data,
#                                    use_alignment=USE_ALIGNMENT, split='validation')
# print(f'train: {len(train_examples)} | val: {len(val_examples)}')

In [5]:
USE_ALIGNMENT = True   # False -> быстрый text-only прогон
MIX_FLEURS    = False  # True -> добавить FLEURS к M-AILABS
LIMIT_TRAIN   = 4000   # None = весь train; аудиокниги крупные, начните умеренно
LIMIT_VAL     = 800

train_examples = build_examples(cfg.data, split="train",      limit=LIMIT_TRAIN,
                                use_alignment=USE_ALIGNMENT, source="ruls", mix_fleurs=MIX_FLEURS)
val_examples   = build_examples(cfg.data, split="validation", limit=LIMIT_VAL,
                                use_alignment=USE_ALIGNMENT, source="ruls", mix_fleurs=MIX_FLEURS)

print(f"train: {len(train_examples)} | val: {len(val_examples)}")
ex = train_examples[0]
print("слова:", ex.words[:12])
print("есть акустика:", ex.has_acoustic, "| форма:", ex.acoustic.shape)

# распределение классов пунктуации — убедимся, что ? ! … теперь присутствуют
from collections import Counter
c = Counter(x for e in train_examples for x in e.punct_ids)
print("распределение пунктуации:", {PUNCT_LABELS[k]: c[k] for k in sorted(c)})
par = Counter(x for e in train_examples for x in e.para_ids)
print("абзацы (NO_PARA/PARA):", {PARA_LABELS[k]: par[k] for k in sorted(par)})

[ruls-archive] распаковка (несколько минут) ...
[ruls-archive] ошибка распаковки: Compressed file ended before the end-of-stream marker was reached


Resolving data files:   0%|          | 0/24 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/24 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/22 [00:00<?, ?it/s]

[load_ruls] HF: istupakov/russian_librispeech split=train (4000 строк).


подготовка [train] + align:   0%|          | 0/4000 [00:00<?, ?клип/s]

In [ ]:
# ПРОВЕРКА: реально ли загрузился M-AILABS (а не demo-fallback на 2 примерах).
# Если train < ~50 примеров — корпус НЕ загрузился, метрики будут бессмысленны.
assert len(train_examples) >= 50, (
    f"M-AILABS не загрузился (train={len(train_examples)}). Это demo-fallback!\n"
    "Укажите рабочее имя датасета в cfg.data.mailabs_repos и перезапустите ячейку выше.\n"
    "Найти имя: huggingface.co/datasets?search=m-ailabs"
)
print(f"OK: загружено {len(train_examples)} train / {len(val_examples)} val примеров.")

OK: загружено 200 train / 50 val примеров.


### (опц.) Пересчёт нормализации акустики на train
Если включена акустика, грубые `ACOUSTIC_NORM` лучше заменить реальными mean/std.

In [ ]:
# from modules.data import compute_acoustic_stats
# import pprint; pprint.pprint(compute_acoustic_stats(train_examples))

## 3. Baseline №1 — BiLSTM
Обучение использует Focal Loss и автовеса классов (передаём `train_examples`).

In [ ]:
vocab = WordVocab.build(train_examples, min_freq=1, max_size=50000)
print("словарь:", len(vocab))

collate = partial(baseline_collate, pad_id=vocab.pad_id)
train_loader = DataLoader(BaselineDataset(train_examples, vocab, cfg.train.max_len),
                          batch_size=cfg.train.batch_size, shuffle=True, collate_fn=collate)
val_loader   = DataLoader(BaselineDataset(val_examples, vocab, cfg.train.max_len),
                          batch_size=cfg.train.batch_size, shuffle=False, collate_fn=collate)

lstm = build_model("lstm", vocab_size=len(vocab), pad_id=vocab.pad_id, use_acoustic=True)
print("параметров:", sum(p.numel() for p in lstm.parameters()))

словарь: 1511
параметров: 3027595


In [ ]:
cfg.train.epochs = 8
cfg.train.lr = 1e-3
lstm = train_model(lstm, train_loader, cfg.train, val_loader=val_loader,
                   eval_fn=evaluate, train_examples=train_examples)  # <-- train_examples для автовесов
lstm_metrics = evaluate(lstm, val_loader, torch.device(DEVICE))
print(pretty_report(lstm_metrics))

[train] loss=focal (gamma=2.0), auto_class_weights=True


эпоха 1/8:   0%|          | 0/13 [00:00<?, ?batch/s]

оценка:   0%|          | 0/4 [00:00<?, ?batch/s]

[epoch 1/8] train_loss=0.3736  val_punct_F1=0.0000


эпоха 2/8:   0%|          | 0/13 [00:00<?, ?batch/s]

...

эпоха 8/8:   0%|          | 0/13 [00:00<?, ?batch/s]

оценка:   0%|          | 0/4 [00:00<?, ?batch/s]

[epoch 8/8] train_loss=0.0502  val_punct_F1=0.1735
[train] загружены лучшие веса (val_punct_F1=0.1761).


оценка:   0%|          | 0/4 [00:00<?, ?batch/s]

ВНИМАНИЕ: accuracy на этой задаче обманчива — класс «нет знака» (O)
преобладает (~80-90%), поэтому ориентируйтесь на recall/F1 по классам
знаков и на macro-F1 (он считается только по присутствующим классам).

=== PUNCT (macro-F1=0.176 по 4 присутствующим классам, acc=0.823) ===
class              P       R      F1   support
O              0.836   0.991   0.907       551
COMMA          0.000   0.000   0.000        96
PERIOD         0.646   0.775   0.705        40
QUESTION       0.000   0.000   0.000         6
EXCLAM         0.000   0.000   0.000         8
ELLIPSIS       0.000   0.000   0.000         0  (нет в данных)

=== PARA (macro-F1=1.000 по 0 присутствующим классам, acc=1.000) ===
class              P       R      F1   support
NO_PARA        1.000   1.000   1.000       701
PARA           0.000   0.000   0.000         0  (нет в данных)

=== CAP (macro-F1=0.037 по 1 присутствующим классам, acc=0.780) ===
class              P       R      F1   support
LOWER          0.782   0.996   0.

## 4. Baseline №2 — Transformer с нуля

In [ ]:
transformer = build_model("transformer", vocab_size=len(vocab), pad_id=vocab.pad_id, use_acoustic=True)
print("параметров:", sum(p.numel() for p in transformer.parameters()))

cfg.train.epochs = 10   # трансформеру с нуля нужно больше эпох/данных
cfg.train.lr = 3e-4
transformer = train_model(transformer, train_loader, cfg.train, val_loader=val_loader,
                          eval_fn=evaluate, train_examples=train_examples)
tr_metrics = evaluate(transformer, val_loader, torch.device(DEVICE))
print(pretty_report(tr_metrics))

параметров: 3554187
[train] loss=focal (gamma=2.0), auto_class_weights=True


эпоха 1/10:   0%|          | 0/13 [00:00<?, ?batch/s]

оценка:   0%|          | 0/4 [00:00<?, ?batch/s]

[epoch 1/10] train_loss=0.4008  val_punct_F1=0.0143


c:\Users\Roman\Documents\Projects\STT_russian_lang\venv\Lib\site-packages\torch\nn\modules\transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\NestedTensorImpl.cpp:182.)
  output = torch._nested_tensor_from_mask(


...

эпоха 10/10:   0%|          | 0/13 [00:00<?, ?batch/s]

оценка:   0%|          | 0/4 [00:00<?, ?batch/s]

[epoch 10/10] train_loss=0.0598  val_punct_F1=0.0051
[train] загружены лучшие веса (val_punct_F1=0.0190).


оценка:   0%|          | 0/4 [00:00<?, ?batch/s]

ВНИМАНИЕ: accuracy на этой задаче обманчива — класс «нет знака» (O)
преобладает (~80-90%), поэтому ориентируйтесь на recall/F1 по классам
знаков и на macro-F1 (он считается только по присутствующим классам).

=== PUNCT (macro-F1=0.019 по 4 присутствующим классам, acc=0.786) ===
class              P       R      F1   support
O              0.790   0.993   0.880       551
COMMA          0.444   0.042   0.076        96
PERIOD         0.000   0.000   0.000        40
QUESTION       0.000   0.000   0.000         6
EXCLAM         0.000   0.000   0.000         8
ELLIPSIS       0.000   0.000   0.000         0  (нет в данных)

=== PARA (macro-F1=1.000 по 0 присутствующим классам, acc=1.000) ===
class              P       R      F1   support
NO_PARA        1.000   1.000   1.000       701
PARA           0.000   0.000   0.000         0  (нет в данных)

=== CAP (macro-F1=0.013 по 1 присутствующим классам, acc=0.775) ===
class              P       R      F1   support
LOWER          0.779   0.993   0.

## 5. Предобученные — RuBERT-base / rubert-tiny2

**Важно:** в первые 2-3 эпохи val-F1 может быть равен 0. Это не поломка — при дообучении с warmup + focal loss голове нужно несколько эпох, чтобы сойти с инициализации (где она предсказывает только доминирующий класс `O`). К 8-12 эпохе F1 выходит на плато. Если после ~12 эпох всё ещё 0 — проверьте, что корпус реально загрузился (ячейка-проверка выше).

In [ ]:
PRESET = "rubert-base"   # или "rubert-tiny2" (легче/быстрее)
model_name = PRETRAINED_PRESETS[PRESET]
print("модель:", model_name)

hf_tok = load_hf_tokenizer(model_name)
ptr_collate = partial(pretrained_collate, pad_id=hf_tok.pad_token_id or 0)
ptr_train_loader = DataLoader(PretrainedDataset(train_examples, hf_tok, cfg.train.max_len),
                              batch_size=cfg.train.batch_size, shuffle=True, collate_fn=ptr_collate)
ptr_val_loader   = DataLoader(PretrainedDataset(val_examples, hf_tok, cfg.train.max_len),
                              batch_size=cfg.train.batch_size, shuffle=False, collate_fn=ptr_collate)

pretrained = build_model("pretrained", model_name=model_name, use_acoustic=True)

модель: DeepPavlov/rubert-base-cased


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
cfg.train.epochs = 10   # RuBERT-голове нужно ~8-12 эпох: первые 2-3 эпохи
                         # F1 может быть 0 (warmup + focal), это НОРМАЛЬНО — не пугайтесь.
pretrained = train_model(pretrained, ptr_train_loader, cfg.train, val_loader=ptr_val_loader,
                         is_pretrained=True, eval_fn=evaluate, train_examples=train_examples)
ptr_metrics = evaluate(pretrained, ptr_val_loader, torch.device(DEVICE))
print(pretty_report(ptr_metrics))

[train] loss=focal (gamma=2.0), auto_class_weights=True


эпоха 1/10:   0%|          | 0/13 [00:00<?, ?batch/s]

оценка:   0%|          | 0/4 [00:00<?, ?batch/s]

[epoch 1/10] train_loss=0.2773  val_punct_F1=0.1484


эпоха 2/10:   0%|          | 0/13 [00:00<?, ?batch/s]

KeyboardInterrupt: 

### Сравнение моделей

In [ ]:
import pandas as pd
rows = [{"модель": n,
         "punct F1": round(m.get("punct_f1_macro", 0), 3),
         "para F1":  round(m.get("para_f1_macro", 0), 3),
         "cap F1":   round(m.get("cap_f1_macro", 0), 3)}
        for n, m in [("BiLSTM", lstm_metrics), ("Transformer", tr_metrics), (PRESET, ptr_metrics)]]
pd.DataFrame(rows)

,модель,punct F1,para F1,cap F1
0,BiLSTM,0.180,1.0,0.038
1,Transformer,0.024,1.0,0.000
2,rubert-base,0.549,1.0,0.498


## 6. Инференс

In [ ]:
restorer = PunctuationRestorer(lstm, kind="lstm", vocab=vocab, device=DEVICE)
# предобученная: PunctuationRestorer(pretrained, kind="pretrained", hf_tokenizer=hf_tok, device=DEVICE)

print(restorer.restore("привет как дела я давно тебя не видел"))
print(restorer.restore("что это было невероятно я не ожидал такого поворота событий"))

Привет как дела я давно тебя не видел
Что это было невероятно я не ожидал такого поворота событий


## 7. Встраивание в SpeechToText-пайплайн
Whisper отдаёт слова + тайм-коды; из них считаются те же акустические признаки, что при обучении (паузы → границы, F0 → `?`/`!`).

In [ ]:
pipeline = STTPunctuationPipeline(restorer)

words = "что это было невероятно я не ожидал такого".split()
t, word_ts = 0.0, []
for w in words:
    word_ts.append({"word": w, "start": round(t,2), "end": round(t+0.3,2)})
    t += 0.3 + (0.7 if w in ("было","невероятно","такого") else 0.05)

print("С паузами:", pipeline({"words": words, "word_timestamps": word_ts, "audio": None}))
print("Текст    :", pipeline({"text": " ".join(words)}))

С паузами: Что это было невероятно я не ожидал такого
Текст    : Что это было невероятно я не ожидал такого


### Реальный Whisper
```python
import whisper, soundfile as sf
asr = whisper.load_model("large-v3")
res = asr.transcribe("audio.wav", language="ru", word_timestamps=True)
words, word_ts = [], []
for seg in res["segments"]:
    for w in seg["words"]:
        tok = w["word"].strip(); words.append(tok)
        word_ts.append({"word": tok, "start": w["start"], "end": w["end"]})
audio, sr = sf.read("audio.wav")
final = STTPunctuationPipeline(restorer)({"words": words, "word_timestamps": word_ts, "audio": audio}, sr=sr)
```

## 8. Настройка против дисбаланса (если знаки всё ещё редки)
Все рычаги в `cfg.train` (модуль `config.py`):
```python
cfg.train.loss_type = "focal"      # "ce" — обычный взвешенный CrossEntropy
cfg.train.focal_gamma = 2.0        # больше -> сильнее фокус на редких знаках (попробуйте 3.0)
cfg.train.auto_class_weights = True  # автовеса по частоте в train
```
И помните: **смотрите recall по знакам и macro-F1, а не accuracy.**

## 9. Сохранение
```python
torch.save(lstm.state_dict(), "lstm_punct.pt"); vocab.save("vocab.json")
torch.save(pretrained.state_dict(), "rubert_punct.pt")
```